# Load Laptop and Review DataFrames

This notebook loads the `products_laptop` and `reviews_laptop` tables from the PostgreSQL database into pandas DataFrames.

In [1]:
import psycopg2
import pandas as pd
import sys
import json
import ast
import re
import os
import csv
from io import StringIO


In [2]:
# --- Database Connection Details ---
DB_NAME = "amazon_electronics_rag"
DB_USER = "akshat"
DB_PASS = "2139"
DB_HOST = "localhost"
DB_PORT = "5432"

In [9]:
def load_data():
    """
    Connects to the PostgreSQL database and loads products_laptop and reviews_laptop
    tables into pandas DataFrames.
    """
    conn = None
    try:
        print(f"Connecting to database '{DB_NAME}' on {DB_HOST}...")
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASS,
            host=DB_HOST,
            port=DB_PORT
        )
        print("Connection successful.")

        # Load products_laptop
        print("Loading 'products_laptop' table...")
        
        query_products_det = "SELECT parent_asin,details FROM products_laptop WHERE price IS NOT NULL AND NOT 'Laptop Travel Accessories' = ANY(categories);"
        df_products_det = pd.read_sql_query(query_products_det, conn)
        print(f"Loaded {len(df_products_det)} rows from 'products_laptop'.")

        query_products = """
                        SELECT parent_asin,title,store, price,categories, average_rating, rating_number,blair_embedding
                        FROM products_laptop
                        WHERE price IS NOT NULL AND NOT 'Laptop Travel Accessories' = ANY(categories);
                        """
        df_products = pd.read_sql_query(query_products, conn)
        print(f"Loaded {len(df_products)} rows from 'products_laptop'.")

        
        return df_products, df_products_det

    except (Exception, psycopg2.Error) as error:
        print(f"Error: {error}", file=sys.stderr)
        return None, None

    finally:
        if conn:
            conn.close()
            print("\nPostgreSQL connection closed.")

In [11]:
df_products, df_products_det = load_data()

Connecting to database 'amazon_electronics_rag' on localhost...
Connection successful.
Loading 'products_laptop' table...
Loaded 2074 rows from 'products_laptop'.


/tmp/ipykernel_21410/3806412340.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_products_det = pd.read_sql_query(query_products_det, conn)
/tmp/ipykernel_21410/3806412340.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_products = pd.read_sql_query(query_products, conn)


Loaded 2074 rows from 'products_laptop'.

PostgreSQL connection closed.


In [12]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2074 entries, 0 to 2073
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   parent_asin      2074 non-null   object 
 1   title            2074 non-null   object 
 2   store            2074 non-null   object 
 3   price            2074 non-null   float64
 4   categories       2074 non-null   object 
 5   average_rating   2074 non-null   float64
 6   rating_number    2074 non-null   int64  
 7   blair_embedding  2074 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 129.8+ KB


In [13]:
import pandas as pd

def get_rated_category(row):
    avg = row['average_rating']
    count = row['rating_number']
    
    # CASE WHEN average_rating > 4 AND average_rating < 6 AND rating_number > 100 THEN 'Highly'
    if 4 < avg < 6 and count > 100:
        return 'Highly'
    
    # WHEN average_rating < 4 AND average_rating > 3 AND rating_number <= 100 THEN 'Well'
    elif 3 < avg < 4 and count <= 100:
        return 'Well'
    
    # ELSE 'Decent'
    else:
        return 'Decent'

# Apply the function row by row (axis=1)
df_products['rated'] = df_products.apply(get_rated_category, axis=1)

# Check the results
print(df_products['rated'].value_counts())
display(df_products[['average_rating', 'rating_number', 'rated']].head())

rated
Highly    908
Decent    760
Well      406
Name: count, dtype: int64


,average_rating,rating_number,rated
0,3.9,98,Well
1,3.5,63,Well
2,3.5,38,Well
3,3.7,72,Well
4,3.7,427,Decent


In [14]:
def parse_postgres_array(array_str):
    if pd.isna(array_str) or not isinstance(array_str, str):
        return []
    
    content = array_str.strip('{}')
    if not content:
        return []
    
    reader = csv.reader(StringIO(content), delimiter=',', quotechar='"')
    try:
        return next(reader)
    except StopIteration:
        return []

# Apply parsing
if 'categories' in df_products.columns:
    sample_val = df_products['categories'].iloc[0] if len(df_products) > 0 else None
    
    if isinstance(sample_val, str) and sample_val.startswith('{'):
        print("Detected Postgres array string format. Parsing...")
        df_productsdf_products['Category_List'] = df_products['categories'].apply(parse_postgres_array)
    elif isinstance(sample_val, list):
        print("Already a list format.")
        df_products['Category_List'] = df_products['categories']
    else:
        df_products['Category_List'] = df_products['categories']
    df_products['Leaf_Category'] = df_products['Category_List'].apply(lambda x: x[-1] if x and len(x) > 0 else None)
    
    print("Category Cleaning Sample:")
    display(df_products[['categories', 'Category_List', 'Leaf_Category']].head())

Already a list format.
Category Cleaning Sample:


,categories,Category_List,Leaf_Category
0,"[Electronics, Computers & Accessories, Compute...","[Electronics, Computers & Accessories, Compute...",Traditional Laptops
1,"[Electronics, Computers & Accessories, Compute...","[Electronics, Computers & Accessories, Compute...",2 in 1 Laptops
2,"[Electronics, Computers & Accessories, Compute...","[Electronics, Computers & Accessories, Compute...",Traditional Laptops
3,"[Electronics, Computers & Accessories, Compute...","[Electronics, Computers & Accessories, Compute...",Traditional Laptops
4,"[Electronics, Computers & Accessories, Compute...","[Electronics, Computers & Accessories, Compute...",Traditional Laptops


In [8]:
df_products_det.head()

,parent_asin,details
0,B00007FV2Z,"{'OS': None, 'GPU': None, 'RAM': None, 'Size':..."
1,B00H7O3O1O,"{'OS': None, 'GPU': None, 'RAM': '4 GB DDR3L S..."
2,B000FOG604,"{'OS': None, 'GPU': None, 'RAM': '1 GB DDR2', ..."
3,B000UDISEW,"{'OS': None, 'GPU': None, 'RAM': '1 GB DDR2', ..."
4,B001GN5OLK,"{'OS': None, 'GPU': None, 'RAM': '4 GB DDR3', ..."


In [15]:
def parse_details(x):
    if isinstance(x, dict):
        return x
    if pd.isna(x) or x == "":
        return {}
    try:
        return json.loads(x)
    except:
        try:
            return ast.literal_eval(x)
        except:
            return {}
df_products_det['details'] = df_products_det['details'].apply(parse_details)

In [16]:
details_expanded = pd.json_normalize(df_products_det['details'])
df_final = pd.concat([df_products_det[['parent_asin']], details_expanded], axis=1)
df_final.head()

,parent_asin,OS,GPU,RAM,Size,Brand,Color,Model,Shape,Style,...,Best Sellers Rank.Laptop Computers,Best Sellers Rank.Computer Monitors,Best Sellers Rank.Fabric Deodorizer,Best Sellers Rank.Unique Electronics,Best Sellers Rank.Internal Hard Drives,Best Sellers Rank.Computers & Accessories,Best Sellers Rank.Traditional Laptop Computers,Best Sellers Rank.Climate Pledge Friendly: Computers,Best Sellers Rank.Climate Pledge Friendly: Electronics,Best Sellers Rank
0,B00007FV2Z,None,None,None,None,Dana,black,None,None,Modern,...,NaN,NaN,NaN,NaN,NaN,NaN,3773.0,NaN,NaN,NaN
1,B00H7O3O1O,None,None,4 GB DDR3L SDRAM,None,HP,Sparkling Black,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B000FOG604,None,None,1 GB DDR2,None,HP,Black,None,None,None,...,NaN,NaN,NaN,NaN,NaN,196818.0,46257.0,NaN,NaN,NaN
3,B000UDISEW,None,None,1 GB DDR2,None,Lenovo,Black,None,None,None,...,NaN,NaN,NaN,NaN,NaN,141913.0,32051.0,NaN,NaN,NaN
4,B001GN5OLK,None,None,4 GB DDR3,None,Dell,Black,None,None,None,...,NaN,NaN,NaN,NaN,NaN,52863.0,9937.0,NaN,NaN,NaN


In [17]:
df_final.dropna(axis=1, how='all', inplace=True)
limit = len(df_final) * 0.2
df_cleaned = df_final.dropna(thresh=limit, axis=1)
print(f"Original columns: {df_final.shape[1]}")
print(f"Cleaned columns: {df_cleaned.shape[1]}")

Original columns: 115
Cleaned columns: 46


In [18]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2074 entries, 0 to 2073
Data columns (total 46 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   parent_asin                                           2074 non-null   object 
 1   RAM                                                   1980 non-null   object 
 2   Brand                                                 2069 non-null   object 
 3   Color                                                 1677 non-null   object 
 4   Series                                                1862 non-null   object 
 5   Voltage                                               1107 non-null   object 
 6   Batteries                                             1679 non-null   object 
 7   CPU Model                                             1895 non-null   object 
 8   Processor                                             1953

In [13]:
# Merge the two dataframes on the primary key
df_master = pd.merge(df_products, df_cleaned, on='parent_asin', how='inner')

# Verify the result
print(f"Product Table Shape: {df_products.shape}")
print(f"Cleaned Det Shape:   {df_cleaned.shape}")
print(f"Master Table Shape:  {df_master.shape}")

# Optional: Check for duplicate columns
# (If both tables had a 'price' column, you'll see 'price_x' and 'price_y')
print("\nColumns in new DataFrame:")
print(df_master.columns.tolist())

Product Table Shape: (2074, 9)
Cleaned Det Shape:   (2074, 46)
Master Table Shape:  (2074, 54)

Columns in new DataFrame:
['parent_asin', 'title', 'store', 'price', 'average_rating', 'rating_number', 'blair_embedding', 'rated', 'Leaf_Category', 'RAM', 'Brand', 'Color', 'Series', 'Voltage', 'Batteries', 'CPU Model', 'Processor', 'Hard Drive', 'Model Name', 'Item Weight', 'Screen Size', 'Memory Speed', 'Power Source', 'Chipset Brand', 'Wireless Type', 'Hard Disk Size', 'Processor Brand', 'Special Feature', 'Card Description', 'Operating System', 'Flash Memory Size', 'Hardware Platform', 'Item model number', 'Screen Resolution', 'Optical Drive Type', 'Product Dimensions', 'Computer Memory Type', 'Date First Available', 'Graphics Coprocessor', 'Hard Drive Interface', 'Number of Processors', 'Max Screen Resolution', 'Graphics Card Ram Size', 'Item Dimensions  LxWxH', 'Number of USB 2.0 Ports', 'Number of USB 3.0 Ports', 'Graphics Card Description', 'Ram Memory Installed Size', 'Hard Drive R

In [14]:
import re

def clean_ram(x):
    if pd.isna(x): return None, None
    x = str(x).upper()

    size_match = re.search(r'(\d+)\s*(GB|MB)', x)
    size = f"{size_match.group(1)}{size_match.group(2)}" if size_match else None
    type_match = re.search(r'(DDR\d\w*)', x)
    ram_type = type_match.group(1) if type_match else None
    
    return size, ram_type
if 'RAM' in df_master.columns:
    df_master[['RAM_Size', 'RAM_Type']] = df_master['RAM'].apply(lambda x: pd.Series(clean_ram(x)))
    print("RAM Cleaning Sample:")
    display(df_master[['RAM', 'RAM_Size', 'RAM_Type']].dropna().head())

RAM Cleaning Sample:


,RAM,RAM_Size,RAM_Type
1,4 GB DDR3L SDRAM,4GB,DDR3L
2,1 GB DDR2,1GB,DDR2
3,1 GB DDR2,1GB,DDR2
4,4 GB DDR3,4GB,DDR3
6,1 GB DDR2,1GB,DDR2


In [15]:
def clean_storage(x):
    if pd.isna(x): return None, None
    x = str(x).upper()
    
    # Extract Size
    size_match = re.search(r'(\d+)\s*(GB|TB)', x)
    size = f"{size_match.group(1)}{size_match.group(2)}" if size_match else None
    
    # Extract Type
    storage_type = "HDD"
    if "SSD" in x or "SOLID STATE" in x:
        storage_type = "SSD"
    elif "EMMC" in x:
        storage_type = "eMMC"
    
    return size, storage_type

# Apply Storage cleaning
if 'Hard Drive' in df_master.columns:
    df_master[['Storage_Size', 'Storage_Type']] = df_master['Hard Drive'].apply(lambda x: pd.Series(clean_storage(x)))
    print("Storage Cleaning Sample:")
    display(df_master[['Hard Drive', 'Storage_Size', 'Storage_Type']].dropna().head())

Storage Cleaning Sample:


,Hard Drive,Storage_Size,Storage_Type
1,64 GB SSD,64GB,SSD
2,80 GB,80GB,HDD
3,160 GB IDE,160GB,HDD
4,300 GB HDD,300GB,HDD
6,40 GB HDD,40GB,HDD


In [17]:
def clean_screen_size(x):
    if pd.isna(x): return None
    x = str(x).lower()
    
    # Extract number
    match = re.search(r'(\d+(\.\d+)?)', x)
    if match:
        try:
            return float(match.group(1))
        except:
            return None
    return None

def get_screen_category(size_inches):
    if pd.isna(size_inches): 
        return None
    
    if size_inches > 16:
        return 'Huge'
    elif size_inches > 14:
        return 'Normal'
    else:
        return 'Compact'  


if 'Screen Size' in df_master.columns:
    df_master['Screen_Size_Inches'] = df_master['Screen Size'].apply(clean_screen_size)
    df_master['Screen_Category'] = df_master['Screen_Size_Inches'].apply(get_screen_category)
    
    print("Screen Size Cleaning Sample:")
    display(df_master[['Screen Size', 'Screen_Size_Inches', 'Screen_Category']].dropna().head())

Screen Size Cleaning Sample:


,Screen Size,Screen_Size_Inches,Screen_Category
1,11.6 Inches,11.6,Compact
2,14.1 Inches,14.1,Normal
3,14.1 Inches,14.1,Normal
4,14.1 Inches,14.1,Normal
5,14.1 Inches,14.1,Normal


In [18]:
def clean_brand(x):
    if pd.isna(x): return "Unknown"
    x = str(x).strip().title()
    
    # Synonyms map
    mapping = {
        "Hewlett Packard": "HP",
        "Hp": "HP",
        "Dell Computer": "Dell",
        "Lenovo Group": "Lenovo",
        "Asus Computer": "ASUS",
        "Asus": "ASUS",
        "Msi": "MSI",
        "Apple Computer": "Apple"
    }
    return mapping.get(x, x)

# Apply Brand cleaning
if 'Brand' in df_master.columns:
    df_master['Brand_Normalized'] = df_master['Brand'].apply(clean_brand)
    print("Brand Cleaning Sample:")
    display(df_master[['Brand', 'Brand_Normalized']].drop_duplicates().head(10))

Brand Cleaning Sample:


,Brand,Brand_Normalized
0,Dana,Dana
1,HP,HP
3,Lenovo,Lenovo
4,Dell,Dell
7,HEWLETT PACKARD,HP
9,None,Unknown
11,Toughbook,Toughbook
32,SAMSUNG,Samsung
35,Acer,Acer
45,ASUS,ASUS


In [19]:

def clean_weight(x):
    if pd.isna(x): return None
    x = str(x).lower()
    
    match = re.search(r'(\d+(\.\d+)?)\s*(pounds|lbs|kg|kgs|ounces|oz)', x)
    if match:
        val = float(match.group(1))
        unit = match.group(3)
        
        if unit in ['pounds', 'lbs']:
            return round(val * 0.453592, 2)
        elif unit in ['ounces', 'oz']:
            return round(val * 0.0283495, 2)
        return val 
    return None

def get_weight_category(weight):
    if pd.isna(weight): 
        return None
    
    if weight > 1.8:
        return 'Heavy'
    else:
        return 'Light'


if 'Item Weight' in df_master.columns:
    df_master['Weight_kg'] = df_master['Item Weight'].apply(clean_weight)
    df_master['Weight_Category'] = df_master['Weight_kg'].apply(get_weight_category)
    
    print("Weight Cleaning Sample:")
    display(df_master[['Item Weight', 'Weight_kg', 'Weight_Category']].dropna().head())

Weight Cleaning Sample:


,Item Weight,Weight_kg,Weight_Category
0,2.45 pounds,1.11,Light
1,3.28 pounds,1.49,Light
2,4 pounds,1.81,Heavy
3,5.9 pounds,2.68,Heavy
4,5.07 pounds,2.30,Heavy


In [20]:
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2074 entries, 0 to 2073
Data columns (total 63 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   parent_asin                                           2074 non-null   object 
 1   title                                                 2074 non-null   object 
 2   store                                                 2074 non-null   object 
 3   price                                                 2074 non-null   float64
 4   average_rating                                        2074 non-null   float64
 5   rating_number                                         2074 non-null   int64  
 6   blair_embedding                                       2074 non-null   object 
 7   rated                                                 2074 non-null   object 
 8   Leaf_Category                                         2074

In [24]:
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2074 entries, 0 to 2073
Data columns (total 63 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   parent_asin                                           2074 non-null   object 
 1   title                                                 2074 non-null   object 
 2   store                                                 2074 non-null   object 
 3   price                                                 2074 non-null   float64
 4   average_rating                                        2074 non-null   float64
 5   rating_number                                         2074 non-null   int64  
 6   blair_embedding                                       2074 non-null   object 
 7   rated                                                 2074 non-null   object 
 8   Leaf_Category                                         2074

In [25]:
folder_path = '/home/akshat/CSE_573/laptop_hybrid_search/agentic_hybrid_search/graph_db/python_scripts/data'

# Define the filename
file_name = 'products.csv'

# Combine the folder path and filename to create the full file path
full_file_path = os.path.join(folder_path, file_name)

# Ensure the directory exists (create it if it doesn't)
os.makedirs(folder_path, exist_ok=True)

# Save the DataFrame to the specified CSV file
df_master[["parent_asin", "blair_embedding","Brand_Normalized","Model Name"]].to_csv(full_file_path, index=False)

In [23]:
folder_path = '/home/akshat/CSE_573/laptop_hybrid_search/agentic_hybrid_search/graph_db/python_scripts/data'

# Define the filename
file_name = 'products_relation.csv'

# Combine the folder path and filename to create the full file path
full_file_path = os.path.join(folder_path, file_name)

# Ensure the directory exists (create it if it doesn't)
os.makedirs(folder_path, exist_ok=True)

# Save the DataFrame to the specified CSV file
df_master[["parent_asin", "price", "rated", "Leaf_Category", "Color","Chipset Brand","Operating System","Average Battery Life (in hours)","RAM_Size","RAM_Type","Storage_Size","Storage_Type","Screen_Category","Brand_Normalized","Weight_Category"]].to_csv(full_file_path, index=False) # index=False prevents writing the DataFrame index as a column